# Trade Log Visualiser
Reads `logs/trades.db` and plots equity curves, fills, signals, and per-strategy PnL.

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

DB_PATH = "logs/trades.db"

if not Path(DB_PATH).exists():
    raise FileNotFoundError(f"{DB_PATH} not found — run a live/paper session first.")

con = sqlite3.connect(DB_PATH)
print("Connected to", DB_PATH)

## Sessions

In [ ]:
sessions = pd.read_sql("SELECT * FROM sessions ORDER BY started_at", con)
sessions["started_at"] = pd.to_datetime(sessions["started_at"])
sessions["ended_at"] = pd.to_datetime(sessions["ended_at"])
sessions["duration"] = sessions["ended_at"] - sessions["started_at"]
sessions["strategy_names"] = sessions["strategy_names"].apply(json.loads)
sessions

In [ ]:
# Pick which session to inspect (change index or session_id as needed)
SESSION_ID = sessions.iloc[-1]["session_id"]
print("Inspecting session:", SESSION_ID)

## Equity Curve

In [ ]:
snaps = pd.read_sql(
    "SELECT * FROM pnl_snapshots WHERE session_id = ? ORDER BY timestamp",
    con, params=(SESSION_ID,)
)
snaps["timestamp"] = pd.to_datetime(snaps["timestamp"])
snaps["strategy_pnl"] = snaps["strategy_pnl"].apply(json.loads)
snaps["strategy_equity"] = snaps["strategy_equity"].apply(json.loads)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(snaps["timestamp"], snaps["total_equity"], lw=1.5, color="steelblue", label="Total equity")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.set_title(f"Equity curve — session {SESSION_ID[:8]}…")
ax.set_ylabel("USD")
ax.legend()
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Per-Strategy Equity

In [ ]:
if snaps["strategy_equity"].iloc[0]:
    eq_df = pd.DataFrame(snaps["strategy_equity"].tolist(), index=snaps["timestamp"])
    fig, ax = plt.subplots(figsize=(14, 4))
    for col in eq_df.columns:
        ax.plot(eq_df.index, eq_df[col], lw=1.2, label=col)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.set_title("Per-strategy equity")
    ax.set_ylabel("USD")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("No per-strategy equity data recorded.")

## Orders & Fills

In [ ]:
orders = pd.read_sql(
    "SELECT * FROM orders WHERE session_id = ? ORDER BY timestamp",
    con, params=(SESSION_ID,)
)
fills = pd.read_sql(
    "SELECT * FROM fills WHERE session_id = ? ORDER BY timestamp",
    con, params=(SESSION_ID,)
)
orders["timestamp"] = pd.to_datetime(orders["timestamp"])
fills["timestamp"] = pd.to_datetime(fills["timestamp"])

print(f"Orders: {len(orders)}   Fills: {len(fills)}")
fills.head()

In [ ]:
# Overlay fills on the equity curve
buys = fills[fills["direction"] == "BUY"]
sells = fills[fills["direction"] == "SELL"]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Top: equity
axes[0].plot(snaps["timestamp"], snaps["total_equity"], lw=1.5, color="steelblue")
axes[0].set_ylabel("USD")
axes[0].set_title("Equity with fill markers")
axes[0].grid(alpha=0.3)

# Bottom: fill prices per symbol
for sym, grp in fills.groupby("symbol"):
    b = grp[grp["direction"] == "BUY"]
    s = grp[grp["direction"] == "SELL"]
    axes[1].scatter(b["timestamp"], b["fill_price"], marker="^", s=60, label=f"{sym} buy")
    axes[1].scatter(s["timestamp"], s["fill_price"], marker="v", s=60, label=f"{sym} sell")

axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[1].set_ylabel("Fill price")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Commissions & Slippage

In [ ]:
if not fills.empty:
    merged = fills.merge(orders[["order_id", "reference_price"]], on="order_id", how="left")
    merged["slippage"] = (merged["fill_price"] - merged["reference_price"]).abs()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    merged.groupby("symbol")["commission"].sum().plot.bar(ax=axes[0], color="coral")
    axes[0].set_title("Total commission by symbol")
    axes[0].set_ylabel("USD")

    merged.groupby("symbol")["slippage"].mean().plot.bar(ax=axes[1], color="mediumpurple")
    axes[1].set_title("Mean slippage by symbol (|fill − ref|)")
    axes[1].set_ylabel("USD")

    plt.tight_layout()
    plt.show()
else:
    print("No fills recorded for this session.")

## Signal Heatmap

In [ ]:
signals = pd.read_sql(
    "SELECT * FROM signals WHERE session_id = ? ORDER BY timestamp",
    con, params=(SESSION_ID,)
)
signals["timestamp"] = pd.to_datetime(signals["timestamp"])

if not signals.empty:
    pivot = signals.pivot_table(
        index="timestamp", columns="symbol", values="weight", aggfunc="mean"
    )

    fig, ax = plt.subplots(figsize=(14, max(3, len(pivot.columns) * 0.6 + 1)))
    im = ax.imshow(
        pivot.T.values, aspect="auto", cmap="RdYlGn",
        vmin=-1, vmax=1,
        extent=[mdates.date2num(pivot.index[0]), mdates.date2num(pivot.index[-1]),
                -0.5, len(pivot.columns) - 0.5]
    )
    ax.set_yticks(range(len(pivot.columns)))
    ax.set_yticklabels(pivot.columns)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.set_title("Signal weights over time (green=long, red=short)")
    fig.colorbar(im, ax=ax, label="weight")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("No signals recorded for this session.")

## Trade Summary Table

In [ ]:
if not fills.empty:
    summary = fills.groupby("symbol").agg(
        fills=("fill_price", "count"),
        total_qty=("quantity", "sum"),
        avg_price=("fill_price", "mean"),
        total_commission=("commission", "sum"),
    ).round(4)
    display(summary)
else:
    print("No fills recorded for this session.")

In [ ]:
con.close()